#Demo- Build a Movie Recommender using Collaborative Filtering
##**Scenario**
StreamFlix has grown its user base significantly over the past year, expanding into multiple regional markets. Despite having a large content library, the platform suffers from low engagement and high churn rates due to generic recommendations that fail to resonate with individual user tastes. StreamFlix wants to implement a data-driven personalized recommendation engine to improve user experience, increase watch time, and reduce churn.

##**Objectives**
The goal is to develop a proof-of-concept recommender system using user-based collaborative filtering, which:

* Analyzes historical user ratings of movies

* Identifies users with similar taste profiles

* Recommends unseen movies to a user based on what similar users liked

This demonstration will serve as the foundation for a scalable solution StreamFlix can later integrate into its production pipeline.

##Step 1: Load and Explore the Data

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load dataset
df = pd.read_csv("movie_ratings_demo.csv")

##Step 2: Create the User-Movie Rating Matrix
Transform data into a matrix where:

* Rows = users

* Columns = movie titles

* Values = ratings

In [ ]:
# Create user-movie matrix
user_movie_matrix = df.pivot_table(index='user_id', columns='movie_title', values='rating')

##Step 3: Fill Missing Values

Since not every user has rated every movie, fill missing values with 0.

In [ ]:
# Fill NaNs with 0 (or you can use mean/median)
user_movie_filled = user_movie_matrix.fillna(0)

 ## Step 4: Compute User Similarity

Use cosine similarity to find users with similar rating behavior.

In [ ]:
# Compute cosine similarity between users
user_similarity = cosine_similarity(user_movie_filled)
user_similarity_df = pd.DataFrame(user_similarity,
                                   index=user_movie_filled.index,
                                   columns=user_movie_filled.index)

## Step 5: Recommend Movies for a Target User

What we’re doing:

* Find top similar users

* Get movies they rated highly

* Recommend movies not yet seen by the target user

In [ ]:
# Function to recommend movies to a target user
def recommend_movies(target_user, num_recommendations=3):
    similar_users = user_similarity_df[target_user].sort_values(ascending=False)
    similar_users = similar_users.drop(target_user)  # Remove self

    # Get movies watched by similar users
    recommended_movies = pd.Series(dtype=float)
    for user in similar_users.index:
        user_ratings = user_movie_filled.loc[user]
        user_ratings = user_ratings[user_ratings > 0]
        recommended_movies = recommended_movies.add(user_ratings, fill_value=0)

    # Remove movies already watched by the target user
    target_user_rated = user_movie_filled.loc[target_user]
    recommended_movies = recommended_movies.drop(labels=target_user_rated[target_user_rated > 0].index, errors='ignore')

    # Sort and return top N
    return recommended_movies.sort_values(ascending=False).head(num_recommendations)

## Step 6: Interpret the Output

Look at the list of recommended movies for the selected user (e.g., U5) and verify if they make sense based on similar users' interests.

In [ ]:
# Example usage:
print("Recommended movies for user U5:")
print(recommend_movies('U5', num_recommendations=3))